In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pathlib
import sys

sys.path.append(str(pathlib.Path.cwd().parent))

from src.mesh import generate_elliptic_mesh
from src.forward_solver import make_conductivity_example1, generate_cauchy_data
from src.dsm import discretize_laplace_beltrami, compute_dsm_indicator
from src.utils import compute_iou_from_grid

# Phase 2: Classical Direct Sampling Method (DSM)

**Project**: Demystifying Iterative Direct Sampling Methods — From Theory to Code

**Objective**: Implement the standard (non-iterative) DSM as a baseline method for inclusion detection. Demonstrate its capabilities and limitations to motivate the IDSM framework.

---

## Theoretical Foundation (Paper 1, Section 2.2)

### 2.1 Green's Function Representation

For the background operator $A_0 y = -\nabla\cdot(\sigma_0 \nabla y)$, the Green's function $G(x', x)$ satisfies $A_0 G(\cdot, x) = \delta(\cdot - x)$ in $\Omega$. From Paper 1, Eq. (2.6), the **scattering field** $y^s = y_\Omega - y_\emptyset$ has the representation:

$$y^s(x') = \int_\Omega G(x', x) \, B[u](y_\Omega)(x) \, dx \tag{2.6}$$

where $B[u](y) = -\nabla\cdot(u\,\nabla y)$ is the perturbation operator and $u = \sigma - \sigma_0$.

### 2.2 DSM Indicator Function (Eq. 2.8)

The DSM indicator exploits the **near-orthogonality** of the Green's function:

$$\eta(x) = \frac{\langle G(\cdot, x), y_d^s \rangle_{H^\gamma(\Gamma)}}{\|G(\cdot, x)\|_{H^\gamma(\Gamma)}} \tag{2.8}$$

For $x$ far from an inclusion, $G(\cdot, x)$ is nearly orthogonal to $y^s$, making $\eta(x)$ small. Near inclusions, the correlation is large. The fractional Sobolev space $H^\gamma(\Gamma)$ is defined via the Laplace-Beltrami operator $(-\Delta_\Gamma)^\gamma$.

### 2.3 Laplace-Beltrami Operator

On the closed boundary curve $\Gamma$, $-\Delta_\Gamma$ has eigenpairs $(\lambda_k, v_k)$. For a circle of perimeter $L$:
$\lambda_0 = 0$, $\lambda_{2k-1} = \lambda_{2k} = (2\pi k/L)^2$, corresponding to Fourier modes $\cos(2\pi ks/L)$, $\sin(2\pi ks/L)$.

The fractional operator and $H^\gamma$ inner product:
$$(-\Delta_\Gamma)^\gamma f = \sum_{k:\lambda_k>0} \lambda_k^\gamma \langle f, v_k\rangle v_k, \quad \langle f, g\rangle_{H^\gamma} = \langle (-\Delta_\Gamma)^\gamma f, g\rangle_{L^2(\Gamma)}$$

**Discretization**: Generalized eigenproblem $K_\Gamma v = \lambda M_\Gamma v$ via 1D P1 FEM on $\Gamma$.

### 2.4 Numerator via Auxiliary PDE (Eq. 2.9)

The numerator $\zeta(x) := \langle G(\cdot, x), y_d^s \rangle_{H^\gamma(\Gamma)}$ is computed for all $x$ by solving:

$$A_0 \zeta = 0 \text{ in } \Omega, \quad T[A_0]\zeta = (-\Delta_\Gamma)^\gamma y_d^s \text{ on } \Gamma \tag{2.9}$$

This works because $\langle G(\cdot,x), f\rangle_{L^2(\Gamma)} = w(x)$ where $A_0 w = 0$, $T[A_0]w = f$.
Applying $(-\Delta_\Gamma)^\gamma$ to $y_d^s$ as Neumann data yields the $H^\gamma$ inner product.

### 2.5 Denominator Approximation (Eq. 2.10)

Two approaches for $\|G(\cdot, x)\|_{H^\gamma(\Gamma)}$:

1. **Distance** (Eq. 2.10): $\|G(\cdot,x)\|_{H^\gamma} \approx C/d(x,\Gamma)^\gamma$ (exploits Green's function singularity)
2. **Integral** (FreeFEM): $R(x) = \sqrt{\int_\Gamma |x-x'|^{-2}\,ds(x')}$ (more accurate near boundary)

### 2.6 Aggregation

For $L$ Cauchy data pairs: $\zeta = \frac{1}{L}\sum_{\ell=1}^L \zeta_\ell$.

**Denominator** (Eq. 2.10): $\|G(\cdot, x)\|_{H^\gamma(\Gamma)} \approx C / d(x, \Gamma)^\gamma$

In [2]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time

from src.mesh import generate_elliptic_mesh
from src.forward_solver import make_conductivity_example1, generate_cauchy_data
from src.dsm import (
    discretize_laplace_beltrami, compute_scattering_data,
    compute_dsm_indicator, plot_dsm_indicator
)
from src.utils import EXAMPLE1_BOXES

# Generate mesh and data
mesh = generate_elliptic_mesh(n_boundary=256)
sigma_true, u_true = make_conductivity_example1(mesh)

def f1(x, y): return x
def f2(x, y): return y
source_funcs = [f1, f2]

print('Mesh: %d nodes, %d triangles' % (mesh.n_points, mesh.n_triangles))
print('Inclusions: two squares at (0.4, 0.2) and (-0.5, -0.2), sigma=0.3')

Mesh: 8088 nodes, 15918 triangles
Inclusions: two squares at (0.4, 0.2) and (-0.5, -0.2), sigma=0.3


## 1. Laplace-Beltrami Eigenspectrum on $\Gamma$

The $H^\gamma(\Gamma)$ inner product is defined via the spectral decomposition of $-\Delta_\Gamma$:
$$(-\Delta_\Gamma)^\gamma f = \sum_{i: \lambda_i > 0} \lambda_i^\gamma (f, v_i)_{L^2(\Gamma)} v_i$$

For a closed curve, eigenvalues come in pairs (corresponding to $\cos(k\theta)$ and $\sin(k\theta)$ on a circle).

In [3]:
lb_op = discretize_laplace_beltrami(mesh, gamma=0.5, n_eigenvalues=100)

evals = lb_op.eigenvalues
print('Eigenvalues of -Delta_Gamma (first 20):')
for i in range(0, min(20, len(evals)), 2):
    if i+1 < len(evals):
        print('  lambda_%d = %.4f,  lambda_%d = %.4f  (pair ratio: %.6f)' % (
            i, evals[i], i+1, evals[i+1],
            evals[i+1]/evals[i] if evals[i] > 1e-10 else 0))

# Visualize eigenfunctions on boundary
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
bdry_pts = mesh.points[mesh.boundary_nodes]
theta = np.arctan2(bdry_pts[:, 1] / 0.8, bdry_pts[:, 0])

for idx, ax in enumerate(axes.flat):
    k = idx + 1  # Skip k=0 (constant mode)
    if k >= len(evals):
        break
    v = lb_op.eigenvectors[:, k]
    ax.scatter(bdry_pts[:, 0], bdry_pts[:, 1], c=v, cmap='RdBu_r',
               s=10, vmin=-np.max(np.abs(v)), vmax=np.max(np.abs(v)))
    ax.set_aspect('equal')
    ax.set_title('$v_{%d}$, $\\lambda_{%d} = %.2f$' % (k, k, evals[k]))
    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-0.95, 0.95)

plt.suptitle('Eigenfunctions of $-\\Delta_\\Gamma$ on Elliptic Boundary', y=1.02)
plt.tight_layout()
plt.savefig('../figures/02_lb_eigenfunctions.png', dpi=150, bbox_inches='tight')
plt.close()

# Eigenvalue growth
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(range(len(evals)), evals, 'bo', markersize=4)
# Reference: k^2 growth for circle
k_ref = np.arange(1, len(evals)//2 + 1)
lam_ref = k_ref**2 * (2*np.pi / 5.65)**2  # Approx for ellipse perimeter ~5.65
ax.semilogy(np.arange(1, len(lam_ref)*2, 2), lam_ref, 'r--', alpha=0.5,
            label='$k^2 (2\\pi/L)^2$ reference')
ax.set_xlabel('Index $i$')
ax.set_ylabel('$\\lambda_i$')
ax.set_title('Eigenvalues of $-\\Delta_\\Gamma$')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/02_lb_eigenvalues.png', dpi=150, bbox_inches='tight')
plt.close()
print('Eigenspectrum plots saved.')

Eigenvalues of -Delta_Gamma (first 20):
  lambda_0 = 0.0000,  lambda_1 = 1.2271  (pair ratio: 0.000000)
  lambda_2 = 1.2271,  lambda_3 = 4.9091  (pair ratio: 4.000587)
  lambda_4 = 4.9092,  lambda_5 = 11.0484  (pair ratio: 2.250572)
  lambda_6 = 11.0484,  lambda_7 = 19.6487  (pair ratio: 1.778413)
  lambda_8 = 19.6487,  lambda_9 = 30.7152  (pair ratio: 1.563219)
  lambda_10 = 30.7152,  lambda_11 = 44.2547  (pair ratio: 1.440810)
  lambda_12 = 44.2547,  lambda_13 = 60.2756  (pair ratio: 1.362015)
  lambda_14 = 60.2756,  lambda_15 = 78.7877  (pair ratio: 1.307124)
  lambda_16 = 78.7877,  lambda_17 = 99.8022  (pair ratio: 1.266724)
  lambda_18 = 99.8022,  lambda_19 = 123.3322  (pair ratio: 1.235766)


Eigenspectrum plots saved.


## 2. DSM Reconstruction — No Noise

Apply the DSM indicator to clean Cauchy data (Example 1, $L=2$ sources).

In [4]:
# Clean data
cauchy_clean = generate_cauchy_data(mesh, sigma_true, source_funcs, noise_level=0.0)

t0 = time.time()
result_clean = compute_dsm_indicator(
    mesh, cauchy_clean, gamma=0.5, n_grid=201,
    denom_method='integral', n_eigenvalues=100
)
t_clean = time.time() - t0

indicator = result_clean['indicator']
print('DSM (no noise, gamma=0.5):')
print('  Computation time: %.2f s' % t_clean)
print('  Indicator range: [%.4e, %.4e]' % (np.nanmin(indicator), np.nanmax(indicator)))

fig = plot_dsm_indicator(result_clean,
    title='DSM Indicator (no noise, $\\gamma=0.5$)',
    save_path='../figures/02_dsm_clean.png',
    inclusion_boxes=EXAMPLE1_BOXES, cmap='hot')
print('Plot saved.')

DSM (no noise, gamma=0.5):
  Computation time: 1.57 s
  Indicator range: [2.8020e-04, 2.1937e-02]


Plot saved.


## 3. Noise Robustness

Test DSM at noise levels $\varepsilon = 10\%$ and $30\%$.

In [5]:
noise_levels = [0.0, 0.1, 0.3]
results = {}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, eps in zip(axes, noise_levels):
    cauchy = generate_cauchy_data(mesh, sigma_true, source_funcs,
                                  noise_level=eps, rng=np.random.default_rng(42))
    result = compute_dsm_indicator(mesh, cauchy, gamma=0.5, n_grid=201,
                                   denom_method='integral', n_eigenvalues=100)
    results[eps] = result
    
    indicator = result['indicator']
    gx, gy = result['grid_x'], result['grid_y']
    
    im = ax.imshow(indicator, origin='lower', cmap='hot',
                   extent=[gx[0], gx[-1], gy[0], gy[-1]],
                   aspect='equal')
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    for box in EXAMPLE1_BOXES:
        cx, cy = box['center']
        hw = box['half_width']
        rect = plt.Rectangle((cx-hw, cy-hw), 2*hw, 2*hw, lw=2,
                             edgecolor='w', facecolor='none', linestyle='--')
        ax.add_patch(rect)
    
    ax.set_title('$\\varepsilon = %d\\%%$' % (eps * 100))
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    
    print('eps=%.0f%%: range [%.4e, %.4e]' % (
        eps*100, np.nanmin(indicator), np.nanmax(indicator)))

plt.suptitle('DSM Noise Robustness ($\\gamma = 0.5$)', y=1.02)
plt.tight_layout()
plt.savefig('../figures/02_dsm_noise_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Noise comparison saved.')

eps=0%: range [2.8020e-04, 2.1937e-02]


eps=10%: range [2.9326e-04, 2.2028e-02]


eps=30%: range [3.1938e-04, 2.2210e-02]


Noise comparison saved.


## 4. Effect of $\gamma$ Parameter

The parameter $\gamma$ in $H^\gamma(\Gamma)$ controls the smoothness of the inner product:
- $\gamma = 0$: $L^2(\Gamma)$ inner product (no smoothing)
- $\gamma = 0.5$: standard choice (related to DtN map)
- $\gamma = 1.0$: $H^1(\Gamma)$ inner product (strong smoothing)

In [6]:
gamma_values = [0.0, 0.25, 0.5, 0.75, 1.0]

cauchy = generate_cauchy_data(mesh, sigma_true, source_funcs,
                              noise_level=0.0, rng=np.random.default_rng(42))

fig, axes = plt.subplots(1, len(gamma_values), figsize=(4*len(gamma_values), 4))

for ax, gamma in zip(axes, gamma_values):
    result = compute_dsm_indicator(mesh, cauchy, gamma=gamma, n_grid=101,
                                   denom_method='integral', n_eigenvalues=80)
    indicator = result['indicator']
    gx, gy = result['grid_x'], result['grid_y']
    
    im = ax.imshow(indicator, origin='lower', cmap='hot',
                   extent=[gx[0], gx[-1], gy[0], gy[-1]],
                   aspect='equal')
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    for box in EXAMPLE1_BOXES:
        cx, cy = box['center']
        hw = box['half_width']
        rect = plt.Rectangle((cx-hw, cy-hw), 2*hw, 2*hw, lw=1.5,
                             edgecolor='w', facecolor='none', linestyle='--')
        ax.add_patch(rect)
    
    ax.set_title('$\\gamma = %.2f$' % gamma)
    ax.set_xlabel('$x_1$')

plt.suptitle('Effect of $\\gamma$ on DSM Indicator (no noise)', y=1.02)
plt.tight_layout()
plt.savefig('../figures/02_dsm_gamma_effect.png', dpi=150, bbox_inches='tight')
plt.close()
print('Gamma effect comparison saved.')

Gamma effect comparison saved.


## 5. Denominator Method Comparison

Compare two denominator approximation approaches:
1. **Integral method** (FreeFEM style): $R(x) = \sqrt{\int_\Gamma |x - x'|^{-2} \, ds(x')}$
2. **Distance method**: $R(x) = d(x, \Gamma)^{-\gamma}$

In [7]:
cauchy = generate_cauchy_data(mesh, sigma_true, source_funcs,
                              noise_level=0.0, rng=np.random.default_rng(42))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, method, label in zip(axes, ['integral', 'distance'],
                              ['Integral (FreeFEM)', 'Distance $d(x,\\Gamma)^{-\\gamma}$']):
    result = compute_dsm_indicator(mesh, cauchy, gamma=0.5, n_grid=151,
                                   denom_method=method, n_eigenvalues=80)
    indicator = result['indicator']
    gx, gy = result['grid_x'], result['grid_y']
    
    im = ax.imshow(indicator, origin='lower', cmap='hot',
                   extent=[gx[0], gx[-1], gy[0], gy[-1]],
                   aspect='equal')
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    for box in EXAMPLE1_BOXES:
        cx, cy = box['center']
        hw = box['half_width']
        rect = plt.Rectangle((cx-hw, cy-hw), 2*hw, 2*hw, lw=2,
                             edgecolor='w', facecolor='none', linestyle='--')
        ax.add_patch(rect)
    
    ax.set_title('Denominator: %s' % label)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('DSM Denominator Method Comparison ($\\gamma=0.5$, no noise)', y=1.02)
plt.tight_layout()
plt.savefig('../figures/02_dsm_denom_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Denominator comparison saved.')

Denominator comparison saved.


## 6. Limitations of Classical DSM (Paper 1, Section 3 Opening)

The DSM has five fundamental limitations that motivate the IDSM:

1. **Blurry reconstruction**: The indicator $\eta(x)$ provides only qualitative localization — it cannot resolve sharp inclusion boundaries. This is inherent to the Green's function correlation approach.

2. **No shape information**: DSM cannot distinguish between square, circular, or irregular inclusions. The indicator smoothly decays from inclusion centers regardless of geometry.

3. **No coefficient recovery**: DSM indicates *where* inclusions are, but not *what* their conductivity is ($\sigma = 0.3$ vs $\sigma = 0.5$). It gives a relative "probability map," not quantitative values.

4. **No iterative refinement**: DSM is a single-pass method — there is no mechanism to use the initial estimate to improve reconstruction quality.

5. **Limited inclusion type discrimination**: DSM cannot easily distinguish between conductive ($\sigma < \sigma_0$) and insulating ($\sigma > \sigma_0$) inclusions in a single reconstruction.

### How IDSM Overcomes These Limitations (Paper 1, Section 3)

| Limitation | IDSM Innovation |
|---|---|
| Blurry indicator | Iterative refinement sharpens boundaries |
| No coefficient values | Direct coefficient recovery via $B[u]$ operator |
| Single-shot | Quasi-Newton loop with convergence guarantees |
| No shape detail | P0 element-wise reconstruction resolves geometry |
| No type discrimination | Separate conductivity/potential channels |

**Key replacements**:
- $(-\Delta_\Gamma)^\gamma$ $\to$ **regularized DtN map** $\Lambda_\alpha(A)$ (Eq. 3.5)
- Single evaluation $\to$ **iterative quasi-Newton** with DFP/BFG low-rank corrections (Eq. 3.14-3.15)
- Indicator function $\to$ **direct P0 reconstruction** $u_{k+1} = \mathcal{P}(R_k \zeta_k)$

In [8]:
# Summary statistics
print('=== Phase 2 Summary ===')
print()
print('DSM Implementation:')
print('  - Laplace-Beltrami via 1D FEM eigendecomposition')
print('  - Numerator: auxiliary Neumann PDE (Eq. 2.9)')
print('  - Denominator: integral approximation (FreeFEM style)')
print('  - Aggregation: average over L=%d Cauchy data pairs' % len(source_funcs))
print()
print('Results:')
for eps in [0.0, 0.1, 0.3]:
    if eps in results:
        ind = results[eps]['indicator']
        print('  eps=%.0f%%: indicator range [%.4e, %.4e], dynamic range %.1f' % (
            eps*100, np.nanmin(ind), np.nanmax(ind),
            np.nanmax(ind) / (np.nanmin(ind) + 1e-30)))
print()
print('Next: Phase 3 - Iterative DSM (IDSM) with regularized DtN map')

=== Phase 2 Summary ===

DSM Implementation:
  - Laplace-Beltrami via 1D FEM eigendecomposition
  - Numerator: auxiliary Neumann PDE (Eq. 2.9)
  - Denominator: integral approximation (FreeFEM style)
  - Aggregation: average over L=2 Cauchy data pairs

Results:
  eps=0%: indicator range [2.8020e-04, 2.1937e-02], dynamic range 78.3
  eps=10%: indicator range [2.9326e-04, 2.2028e-02], dynamic range 75.1
  eps=30%: indicator range [3.1938e-04, 2.2210e-02], dynamic range 69.5

Next: Phase 3 - Iterative DSM (IDSM) with regularized DtN map


## 7. Implementation Details and IoU Computation

### Laplace-Beltrami Discretization (Paper 1, Eq. 2.8–2.10)

The 1D FEM eigenproblem on the boundary $\Gamma$:
$$K_\Gamma v_i = \lambda_i M_\Gamma v_i$$

yields eigenvalues $0 = \lambda_0 < \lambda_1 \leq \lambda_2 \leq \cdots$ and eigenvectors $\{v_i\}$. The fractional operator is applied as:
$$(-\Delta_\Gamma)^\gamma f = \sum_{i:\lambda_i > 0} \lambda_i^\gamma (f, v_i)_{M_\Gamma} \, v_i$$

### DSM Numerator via Auxiliary PDE

For each Cauchy data pair $\ell$:
1. Compute scattering data: $y_d^{s,\ell} = y_\emptyset^\ell - y_d^\ell$
2. Apply $(-\Delta_\Gamma)^\gamma$ to get the filtered data on $\Gamma$
3. Solve Neumann PDE with this as boundary source $\to \zeta_\ell(x)$
4. Average: $\zeta = \frac{1}{L}\sum_\ell \zeta_\ell$

### Grid-Based IoU with Area-Matched Thresholding

Standard IoU requires a threshold to binarize the DSM indicator. We use **area-matched thresholding**: choose the threshold $\tau$ such that the thresholded region has the same area as the true inclusion support. This eliminates threshold selection bias and enables fair comparison with IDSM P0 reconstructions.

In [9]:
mesh = generate_elliptic_mesh(n_boundary=120)
sigma_true, u_true = make_conductivity_example1(mesh)

def f1(x, y): return x

data = generate_cauchy_data(mesh, sigma_true, [f1], noise_level=0.05)

lb = discretize_laplace_beltrami(mesh, gamma=0.5, n_eigenvalues=40)
print('First 10 LB eigenvalues:', lb.eigenvalues[:10])

dsm = compute_dsm_indicator(mesh, data, gamma=0.5, n_grid=121, n_eigenvalues=40)
iou = compute_iou_from_grid(mesh, u_true, dsm['indicator'], dsm['mask'])
print('DSM grid IoU =', iou)

First 10 LB eigenvalues: [-4.74176256e-13  1.22751150e+00  1.22757312e+00  4.91356987e+00
  4.91362412e+00  1.10684428e+01  1.10684693e+01  1.97092970e+01
  1.97093071e+01  3.08602491e+01]


DSM grid IoU = 0.13672182821118992


## 8. Conductive vs Insulating: DSM Cannot Classify (Paper 1, Section 3)

A fundamental limitation of DSM is its **inability to distinguish between conductive and insulating inclusions**. The DSM indicator $\eta(x)$ measures the *magnitude* of the correlation $|\langle G(\cdot, x), y_d^s \rangle|$, which is large near any inclusion regardless of whether $\sigma > \sigma_0$ (conductive) or $\sigma < \sigma_0$ (insulating).

**Why this happens**: The scattering field $y^s = y_\Omega - y_\emptyset$ is generated by the perturbation operator $B[u](y) = -\nabla\cdot(u\,\nabla y)$. The DSM indicator detects the *support* of $u$ (where $u \neq 0$) but not its *sign* (whether $u > 0$ or $u < 0$). Physically, both a conductor ($\sigma = 3.0$) and an insulator ($\sigma = 0.3$) produce scattering signals, and DSM treats both the same.

This limitation is a key motivation for IDSM, which **directly reconstructs** $u = \sigma - \sigma_0$ and therefore recovers the sign, enabling inclusion type classification (Paper 1, Section 3).

In [10]:
from src.forward_solver import make_conductivity_conductive

mesh_cls = generate_elliptic_mesh(n_boundary=256)

# Insulating: sigma = 0.3 (u = -0.7)
sigma_ins, u_ins = make_conductivity_example1(mesh_cls)

# Conductive: sigma = 3.0 (u = +2.0)
sigma_con, u_con = make_conductivity_conductive(mesh_cls)

cauchy_ins = generate_cauchy_data(mesh_cls, sigma_ins, source_funcs, noise_level=0.0)
cauchy_con = generate_cauchy_data(mesh_cls, sigma_con, source_funcs, noise_level=0.0)

dsm_ins = compute_dsm_indicator(mesh_cls, cauchy_ins, gamma=0.5, n_grid=201,
                                 denom_method='integral', n_eigenvalues=100)
dsm_con = compute_dsm_indicator(mesh_cls, cauchy_con, gamma=0.5, n_grid=201,
                                 denom_method='integral', n_eigenvalues=100)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Insulating DSM
ax = axes[0]
gx, gy = dsm_ins['grid_x'], dsm_ins['grid_y']
im = ax.imshow(dsm_ins['indicator'], origin='lower', cmap='hot',
               extent=[gx[0], gx[-1], gy[0], gy[-1]], aspect='equal')
plt.colorbar(im, ax=ax, shrink=0.8)
for box in EXAMPLE1_BOXES:
    cx, cy = box['center']
    hw = box['half_width']
    rect = plt.Rectangle((cx-hw, cy-hw), 2*hw, 2*hw, lw=2,
                         edgecolor='cyan', facecolor='none', linestyle='--')
    ax.add_patch(rect)
ax.set_title('DSM: Insulating ($\\sigma=0.3$, $u<0$)')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')

# Conductive DSM
ax = axes[1]
im = ax.imshow(dsm_con['indicator'], origin='lower', cmap='hot',
               extent=[gx[0], gx[-1], gy[0], gy[-1]], aspect='equal')
plt.colorbar(im, ax=ax, shrink=0.8)
for box in EXAMPLE1_BOXES:
    cx, cy = box['center']
    hw = box['half_width']
    rect = plt.Rectangle((cx-hw, cy-hw), 2*hw, 2*hw, lw=2,
                         edgecolor='cyan', facecolor='none', linestyle='--')
    ax.add_patch(rect)
ax.set_title('DSM: Conductive ($\\sigma=3.0$, $u>0$)')
ax.set_xlabel('$x_1$')

# Difference
ax = axes[2]
diff = np.abs(dsm_ins['indicator'] - dsm_con['indicator'])
im = ax.imshow(diff, origin='lower', cmap='Greys',
               extent=[gx[0], gx[-1], gy[0], gy[-1]], aspect='equal')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('$|\\eta_{\\mathrm{ins}} - \\eta_{\\mathrm{con}}|$')
ax.set_xlabel('$x_1$')

plt.suptitle('DSM Cannot Distinguish Inclusion Type', y=1.02)
plt.tight_layout()
plt.savefig('../figures/02_dsm_classify.png', dpi=150, bbox_inches='tight')
plt.close()

# Quantitative comparison
ind_ins = dsm_ins['indicator']
ind_con = dsm_con['indicator']
print('DSM classification failure:')
print('  Insulating indicator range: [%.4e, %.4e]' % (np.nanmin(ind_ins), np.nanmax(ind_ins)))
print('  Conductive indicator range: [%.4e, %.4e]' % (np.nanmin(ind_con), np.nanmax(ind_con)))
print('  Both indicators are POSITIVE everywhere => no sign information')
print('  DSM cannot tell sigma=0.3 from sigma=3.0')
print()
print('  Correlation between indicators: %.4f' % np.corrcoef(
    ind_ins[~np.isnan(ind_ins)].ravel(),
    ind_con[~np.isnan(ind_con)].ravel())[0, 1])

DSM classification failure:
  Insulating indicator range: [2.8020e-04, 2.1937e-02]
  Conductive indicator range: [2.4597e-04, 1.8393e-02]
  Both indicators are POSITIVE everywhere => no sign information
  DSM cannot tell sigma=0.3 from sigma=3.0

  Correlation between indicators: 0.9957
